# OWOD — eloszlás-tudatos aktív annotáció és inkrementális tanulás

**Mit mérünk.** A detektor 19 osztályt ismer. A világban 80 van. Adott egy annotációs
keret — mondjuk körönként 600 régió, amit egy ember felcímkéz. **Melyik régiókat kérjük?**
És amikor az új osztály ismertté válik, **mit adunk vissza a régiekből**, hogy ne felejtsen?

A pontszám, amit a kutatási terv javasol, és amit a 2026-08-25-i konzultáció három ponton
átdefiniált:

$$ s(x) \;=\; U(x) \;+\; \lambda\,D(x) \;+\; \gamma\,w(\hat c(x))\cdot \mathrm{coh}(x) $$

| tag | mit mér | a konzultáció után |
|---|---|---|
| $U$ | bizonytalanság — a poszterior entrópiája | **változatlan** |
| $D$ | diverzitás | nem egy fix horgonytól, hanem a **növekvő címkézett halmaztól** való távolság, plusz **batch-diverzitás** |
| $w$ | ritkaság | a jelölt klaszterének mérete — **ugyanabból a klaszterezésből**, amiből $D$ is jön |
| $\mathrm{coh}$ | lokális támogatottság | **bináris kapu**: 0 vagy 1, nem súly |

**Két üzemmód.** `RUN_GPU = False` mellett a notebook a commitolt PROB-átfutáson dolgozik:
80 000 valódi jelöltrégió, GPU és adathalmaz nélkül, pár perc. Ez válaszolja meg,
**mit választ ki egy pontszám**. `RUN_GPU = True` mellett a PROB súlyai ténylegesen
frissülnek, és a lánc végigmegy a task1 → task10 soron. Ez válaszolja meg, **mennyit
felejt és mennyit tanul**.

> **A két üzemmód nem cserélhető fel.** A korábbi munka lemérte: a fagyasztott
> jellemzőkön alapuló szimuláció a felejtés szerint **fordított sorrendbe** rakja az
> akvizíciós módszereket, mint a valódi detektor. Tehát a szimuláció összemérheti, mit
> *választ ki* egy pontszám, de nem állíthatja, hogy egyik arm kevesebbet felejt.

## Paraméterek — csak ezt kell átírni

In [ ]:
# ============================== PARAMETERS ==============================
RUN_GPU = True          # False: frozen pool, CPU, minutes. True: real PROB training.
SMOKE_TEST = False       # GPU only: two short tasks first, to prove the pipeline
                         # works before committing hours. Set False for the real chain.
MINIMAL_CHAIN = False    # GPU only: three incremental tasks instead of five, for
                         # when the CUDA kernel will not build and everything is 3x
                         # slower. Head plus two tail classes; still answers the
                         # plan's question, over a shorter chain.
FAST_CHAIN = True        # GPU only, and only when SMOKE_TEST is False: the shorter
                         # chain that still covers head and all three tail classes.
                         # ~4.3 h per arm instead of ~9.3. See docs/futtatas.md.

# --- the task chain ---------------------------------------------------
N_TASKS               = 10     # task 1 is PROB's pretrained t1.pth; 9 more follow
BUDGET_PER_TASK       = 600    # regions the annotator is asked about, per task
ROUNDS_PER_TASK       = 6      # 1 = 600x1, 6 = 6x100, 12 = 12x50  (consultation, 7)
CANDIDATE_IMAGES      = 4000   # unlabelled images offered to the selector each task
PROPOSALS_PER_IMAGE   = 50     # PROB offers 100. 4000 x 50 proposals is a 400 MB
                               # export; 4000 x 100 is 800 MB and Colab notices.

# --- the experimental variables ---------------------------------------
# This is the replay study: the selection arm is held fixed and the *replay*
# arms are swept, because the question is what a fixed rehearsal budget's class
# composition does to the stability-plasticity trade-off. Every (selection,
# replay) pair writes to its own workspace, and the chain stops cleanly when
# TIME_BUDGET_MINUTES runs out, so one Run all does as much as the session
# allows and the next one continues where it stopped.
#
# `random + none` has already been measured and its workspace is left alone:
# naming an arm here is what schedules it, so the baseline is simply not listed.
ARMS                  = ("random",)             # owl.selection.ARMS
REPLAY_ARMS           = ("uniform", "tail_favouring")   # owl.replay.ARMS
LABELLING_POLICY      = "known_plus_selected"   # box_only | full_image | known_plus_selected
REPLAY_REALLOCATE     = False  # True: re-derive the memory every task instead of keeping it

# --- training (GPU only) ----------------------------------------------
EPOCHS                = 5
LEARNING_RATE         = 2e-4   # PROB's own default. 2e-5 was the earlier bottleneck.
BATCH_SIZE            = 2
EVAL_MAX_PER_CLASS    = 150    # caps evaluation cost; see section 8
EVAL_REMAINDER_RATIO  = 1
TIME_BUDGET_MINUTES   = 420    # stop cleanly before Colab does it for you

# --- the smoke test overrides, applied only when SMOKE_TEST is on ------
SMOKE = dict(N_TASKS=3, BUDGET_PER_TASK=100, ROUNDS_PER_TASK=2,
             CANDIDATE_IMAGES=300, PROPOSALS_PER_IMAGE=50, EPOCHS=1,
             EVAL_MAX_PER_CLASS=8, EVAL_REMAINDER_RATIO=0, TIME_BUDGET_MINUTES=45)

# t2..t6: traffic light (head), fire hydrant, stop sign, parking meter (tail),
# bench (head). Every frequency group the research is about, at half the cost.
FAST = dict(N_TASKS=6, CANDIDATE_IMAGES=2000)

# t2..t4: traffic light (head), fire hydrant and stop sign (tail).
MINIMAL = dict(N_TASKS=4, CANDIDATE_IMAGES=1500)

# --- reproducibility ---------------------------------------------------
SEED                  = 0
N_CLUSTERS            = 1600

# --- GPU paths ---------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/OWL"   # t1.pth and the annotation archives live here
# ========================================================================

# The smoke test only shrinks the GPU chain. The CPU sections cost minutes either
# way, so there is nothing to shrink there and nothing to be gained by doing it.
if RUN_GPU and SMOKE_TEST:
    globals().update(SMOKE)
    print("SMOKE_TEST is on: two short tasks, a tiny evaluation split. This proves the\n"
          "pipeline runs end to end. It does NOT produce a result worth reporting —\n"
          "set SMOKE_TEST = False once it has passed.")
    print({key: globals()[key] for key in SMOKE})
elif RUN_GPU:
    if MINIMAL_CHAIN:
        globals().update(MINIMAL)
        print("MINIMAL_CHAIN is on:", MINIMAL, "— the shortest chain that still "
              "covers head and tail.")
    elif FAST_CHAIN:
        globals().update(FAST)
        print("FAST_CHAIN is on:", FAST, "— about 4.3 h per arm with the compiled "
              "kernel, ~3x that on the fallback.")
    else:
        print("The full chain:", N_TASKS - 1, "tasks, about 9.3 h per arm on a T4. "
              "TIME_BUDGET_MINUTES stops it cleanly and the next run resumes.")


# --- what this Run all will actually do -------------------------------------
# Printed before anything expensive, and derived from the values above rather
# than retyped, so a parameter edited in the Colab UI cannot disagree with the
# run it produces. Values that live in `owl` — the replay budget M, the class
# order, the split name — are deliberately not repeated here; they are reported
# by the cells that import them.
def describe_experiment():
    runs = [f"{arm}__{replay}" for arm in ARMS for replay in REPLAY_ARMS]
    rows = [
        ("selection arm(s)", ", ".join(ARMS)),
        ("replay arm(s)", ", ".join(REPLAY_ARMS)),
        ("tasks", f"{N_TASKS}  (t1 anchor + {N_TASKS - 1} incremental)"),
        ("candidate images / task", f"{CANDIDATE_IMAGES}"),
        ("annotation budget / task", f"{BUDGET_PER_TASK} regions"),
        ("selection rounds", f"{ROUNDS_PER_TASK} x {BUDGET_PER_TASK // ROUNDS_PER_TASK}"),
        ("labelling policy", LABELLING_POLICY),
        ("replay re-allocation", f"{REPLAY_REALLOCATE}"),
        ("epochs / lr / batch", f"{EPOCHS} / {LEARNING_RATE} / {BATCH_SIZE}"),
        ("seed", f"{SEED}"),
        ("clusters", f"{N_CLUSTERS}"),
        ("scheduled GPU runs", f"{len(runs)}"),
        ("max time, shared", f"{TIME_BUDGET_MINUTES} min across all {len(runs)} run(s)"),
    ]
    width = max(len(name) for name, _ in rows)
    print("=" * 72)
    print("THIS RUN")
    print("=" * 72)
    for name, value in rows:
        print(f"  {name:<{width}}  {value}")
    print("  workspaces")
    for run in runs:
        print(f"    {DRIVE_ROOT}/work/{run}")
    if not runs:
        print("    (none: ARMS or REPLAY_ARMS is empty, so nothing is scheduled)")
    print("=" * 72)


describe_experiment()


## Környezet

In [ ]:
import json, os, subprocess, sys, tarfile, time
from pathlib import Path

# --- where owl lives -------------------------------------------------------
# Local: walk up to the checkout. Colab: a *fresh* clone every time.
#
# Re-cloning looks wasteful and is not. `git reset --hard` updates the files on
# disk but Python keeps the modules it already imported, so a cell rerun after a
# repository update runs the old code against the new notebook and fails with
# something that looks nothing like the cause — `unexpected keyword argument`,
# for instance. Deleting the checkout and purging sys.modules is what makes
# Runtime > Run all mean what it says.
OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"

if RUN_GPU:
    from google.colab import drive
    drive.mount("/content/drive")

    ROOT = Path("/content/owod-active")
    subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", OWL_REPOSITORY, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
    print("owl:", subprocess.run(["git", "log", "-1", "--format=%h %s"], cwd=ROOT,
                                 capture_output=True, text=True, check=True).stdout.strip())
else:
    ROOT = Path.cwd()
    while not (ROOT / "owl").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

# Drop any owl module the kernel already holds, so the import below reads the
# files that are on disk right now rather than the ones from before the clone.
for name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[name]

sys.path.insert(0, str(ROOT))
import numpy as np
from owl import (bridge, clustering, evaluation_subset, labelling, metrics,
                 proposals, protocol, replay, runner, scoring, selection)
from owl.runner import table

# Drift guard. The clone is refreshed every run, but these *cells* come from
# whoever last saved the notebook — so the two can disagree, and when they do the
# error surfaces somewhere unrelated. Check here instead, in both directions.
import inspect

_REQUIRED = {
    "runner.run_chain(prepare_images=)": "prepare_images" in inspect.signature(
        runner.run_chain).parameters,
    "runner.CycleConfig.proposals_per_image": "proposals_per_image" in (
        runner.CycleConfig.__dataclass_fields__),
    "runner.CycleConfig.reuse_deferred_labels": "reuse_deferred_labels" in (
        runner.CycleConfig.__dataclass_fields__),
    "evaluation_subset.SHARED_TEST_SET": hasattr(
        evaluation_subset, "SHARED_TEST_SET"),
    "metrics.per_class_ap50": hasattr(metrics, "per_class_ap50"),
}
_missing = sorted(name for name, present in _REQUIRED.items() if not present)
assert not _missing, (
    f"The owl checkout at {ROOT} is older than this notebook and is missing "
    f"{_missing}. This should not happen — the clone is refreshed on every run — "
    "so check that the repository pushed."
)

# The other direction — a notebook older than owl — is handled by not repeating
# anything: every value the two must agree on is read from owl below, never
# retyped here. The split name was the last exception and is now imported.

print("owl from:", ROOT)
print("mode:", "GPU — PROB weights are updated" if RUN_GPU else "CPU — frozen PROB pass")

## 1. A taszk-lánc: egy új osztály taszkonként

Ez a legfontosabb szerkezeti döntés, és a konzultáció kérte így.

A korábbi felállásban egy inkrementális lépés **húsz** új osztályt adott hozzá 600
annotációból — osztályonként ~30 régió. Az eredmény mérhetetlen volt: az új osztályok
mAP50-je **0,010** lett, miközben a teljes t2-felügyelet 36,13-at ér el. A csereárfolyam
(hány régi mAP-pontot fizetünk egy új pontért) **2931** volt a teljes felügyelet 0,20-a
ellenében.

Taszkonként **egy** osztállyal ugyanaz a 600 régió mind egy osztályra megy. Ez az a
változtatás, ami a plaszticitást egyáltalán mérhetővé teszi.

Az osztálysorrend nem szabad: a PROB kiértékelője pozíció szerint indexeli az osztályokat,
tehát csak a hivatalos sorrend prefixét lehet ismertté nyilvánítani. Szerencsére ez a
prefix magától lefedi mind a három gyakorisági csoportot.

In [ ]:
chain = protocol.build_chain(N_TASKS)
print(table(protocol.describe_chain(chain)))

## 2. A jelöltkészlet

`RUN_GPU = False` esetén ez a commitolt PROB-átfutás: az `exps/SOWODB/PROB/t1.pth`
checkpoint egyetlen forward-passe 2 400 benchmark-képen, minden régió a benchmark saját
annotációjához illesztve IoU 0,5-nél. Ez teszi lehetővé, hogy az egész annotációs kör
laptopon fusson.

Amit egy jelölt hordoz: doboz, 256 dimenziós PROB-dekóder-embedding, osztály-poszterior,
objectness. **A kiválasztás ezeken kívül semmit nem lát** — az `oracle()` hívás az egyetlen
hely, ahol címke előkerül, és azt csak a `labelling` modul hívja meg, a keret elköltése
*után*.

In [ ]:
pool = proposals.from_frozen_pool(split="pool")
print(table([pool.describe()]))

oracle = pool.oracle()
groups = protocol.load_groups()
group_of = np.asarray([groups.get(name, "") for name in oracle.class_name])
unknown = oracle.kind == "unknown"
print("\nreal unknown objects in the pool, by frequency group:")
print(table([{g: int(((group_of == g) & unknown).sum()) for g in ("head", "medium", "tail")}]))
print(f"\nonly {unknown.mean():.1%} of proposals sit on a real unknown object — "
      "that is the needle the score has to find.")

## 3. Egy klaszterezés, amiből $D$ és $w$ is kijön

*(konzultáció, 3. pont)*

Ne külön becsüljük a ritkaságot és a diverzitást. Particionáljuk a jelöltkészletet
**egyszer** az embedding-térben, és olvassuk le mind a kettőt ugyanarról a partícióról:

- **ritkaság** = milyen kicsi a jelölt klasztere;
- **diverzitás** = milyen messze van a klasztere a known-tartalmú klaszterektől.

A partíció minőségi mérőszáma nem sziluett-érték, hanem **known-szennyezés**: hány már
ismert elem esik olyan klaszterbe, amit unknown-jelöltnek minősítenénk.

**Ez a diagnosztika orákulum nélkül lefuttatható**, mert a detektor a saját ismert
osztályait maga is fel tudja címkézni — és utólag ellenőrizhető a benchmark annotációi
ellen. A két oszlop különbsége mondja meg, mennyire bízhatunk a becslésben.

Egy részlet, ami könnyen elrontja: „a klaszter többsége known" szabály itt degenerált,
mert a készlet 81%-a háttér, tehát szinte egy klaszter sem ér el 50% knownt. A helyes
kérdés az **dúsulás**: több known van ebben a klaszterben, mint egy véletlenszerűben?

In [ ]:
detector_known = clustering.predicted_known(pool.posterior, len(protocol.TASK1))
truth_known = pool.oracle().kind == "known"
print(f"the detector labels its own known classes with precision "
      f"{float((truth_known & detector_known).sum() / detector_known.sum()):.2f} — "
      "no annotation involved")

rows = []
for k in (200, 400, 800, 1600, 3200):
    part = clustering.fit(pool.embeddings, n_clusters=k, seed=SEED)
    rows.append({
        "n_clusters": k,
        "mean_cluster_size": float(part.sizes[part.sizes > 0].mean()),
        "contamination_estimated": clustering.contamination(part, detector_known)["contamination"],
        "contamination_verified": clustering.contamination(part, truth_known)["contamination"],
        "unknown_recall": clustering.contamination(part, truth_known)["unknown_recall"],
    })
print("\n" + table(rows, digits=3))

partition = clustering.fit(pool.embeddings, n_clusters=N_CLUSTERS, seed=SEED)
print(f"\nusing K = {N_CLUSTERS} for everything below.")

## 4. A koherencia-kapu: bináris DBSCAN

*(konzultáció, 2. pont)*

A kérés: `coh(x) ∈ {0, 1}` — kapcsoló, ne súly. Ha egy jelölt DBSCAN-zajpont, akkor
`coh = 0`, azt nem akarjuk megtanulni.

Az indoklás jó: egy magányos, semmire nem hasonlító régió nagy eséllyel rossz doboz.
**A mérés viszont nem támasztja alá — és a mérés fontosabb.** A kapu a PROB saját
jellemzőterében *gyakrabban* dobja ki a valódi ismeretlen objektumokat, mint a hátteret.

A mechanizmus nem rejtélyes: a készlet 81%-a háttér, a háttérrégiók pedig szinte
egymás másolatai, tehát ők ülnek a tér legsűrűbb részén. Egy sűrű környezetben lenni ebben
a készletben azt jelenti, hogy **hátérnek nézel ki**. A táblázat utolsó oszlopa a lényeg:
a kapu után az ismeretlen objektumok aránya *csökken*.

**Amit ebből tanulunk, és amit a 3. pont old meg:** ha ugyanabból a klaszterezésből jön a
ritkaság is, akkor a hatalmas háttér-klaszter magától ~0 súlyt kap — a `w · coh` szorzat
tehát a *ritkaság* oldalán szűri ki a hátteret, nem a kapu oldalán.

In [ ]:
rows = []
for eps in (0.15, 0.25, 0.35, 0.45):
    part = clustering.fit(pool.embeddings, method="dbscan", eps=eps,
                          min_samples=5, pca_dimensions=32, seed=SEED)
    noise = part.is_noise
    kinds = pool.oracle().kind
    rows.append({
        "eps": eps,
        "clusters": part.n_clusters,
        "noise_background": float(noise[kinds == "background"].mean()),
        "noise_known": float(noise[kinds == "known"].mean()),
        "noise_real_unknown": float(noise[kinds == "unknown"].mean()),
        "unknown_share_before": float((kinds == "unknown").mean()),
        "unknown_share_after_gate": float((kinds[~noise] == "unknown").mean()),
    })
print(table(rows, digits=3))
print("\nRead the last two columns: the gate lowers the share of real unknown objects\n"
      "at every eps. It removes what we are trying to find.")

## 5. A kiválasztás: melyik pontszám mit vesz meg

*(konzultáció, 1. és 7. pont)*

Minden arm **ugyanazt a 600 régiót** kérdezi meg — az orákulum ára azonos. A kérdés az,
hogy mi jön vissza a pénzért. A `rounds` oszlop a 7. pont: egy lépésben adjuk oda a
keretet, vagy 100-asával, közben újraszámolva.

Az armok, és hogy mi az egy változó, amiben eltérnek:

| arm | mi ez |
|---|---|
| `random` | a padló |
| `entropy` | csak $U$ — a klasszikus aktív tanulás |
| `objectness` | **az ingyenes kontroll**: `objectness × √terület`, semmi tanulás, semmi eloszlás |
| `plan` | a kutatási terv egyenlete pontosan úgy, ahogy le van írva |
| `consult` | a konzultáció $D$-je (címkézettektől mért távolság) + bináris kapu |
| `consult_batch` | ugyanaz + batch-diverzitás |
| `prior_consult` | az ingyenes prior **szorzóként**, a konzultáció tagjaival |
| `prior_consult_batch` | ugyanaz + batch-diverzitás — a szintézis |

In [ ]:
rows = []
for name in ("random", "entropy", "objectness", "plan",
             "consult", "consult_batch", "prior_consult", "prior_consult_batch"):
    for rounds in (1, ROUNDS_PER_TASK):
        picked = selection.select(pool, selection.ARMS[name], budget=BUDGET_PER_TASK,
                                  rounds=rounds, n_known=len(protocol.TASK1),
                                  partition=partition)
        index = picked.indices
        found = pool.oracle().kind[index] == "unknown"
        g = group_of[index][found]
        names = pool.oracle().class_name[index][found]
        rows.append({
            "arm": name, "rounds": rounds,
            "unknown_objects": int(found.sum()),
            "head": int((g == "head").sum()),
            "medium": int((g == "medium").sum()),
            "tail": int((g == "tail").sum()),
            "classes": int(np.unique(names).size),
            "tail_classes": int(np.unique(names[g == "tail"]).size),
            "images": int(picked.images(pool).size),
        })
print(table(sorted(rows, key=lambda r: -r["tail"])))

### Mit olvasunk ki ebből

Három dolog, és mind a három ellenőrizhető a fenti táblázatban.

1. **A terv egyenlete önmagában nem veri a randomot.** A `plan` sor az egyenlet pontos
   implementációja, és nagyjából a random szintjén találja meg az ismeretleneket. Ez nem
   implementációs hiba: a `U + λD + γw·coh` három tagja együtt éppen az izolált
   outliereket kedveli, ahogy a terv maga is figyelmeztet rá.

2. **A konzultáció javításai működnek.** `consult` és `consult_batch` a terv verziójának
   a többszörösét találja meg, és a `rounds` növelése *csak* azokon az armokon segít,
   amelyeknek van mit frissíteniük — az `objectness` sor változatlan 1 és 6 kör között,
   a `consult` nem. Ez a 7. pont, lemérve.

3. **A tail oszlop a lényeg, nem az összesen.** Az `objectness` a legtöbb ismeretlent
   találja, de főleg nagy, feltűnő head-objektumokat. A `prior_consult_batch` kevesebb
   ismeretlent talál összesen, viszont **több tail-objektumot** — a keretet a fejről a
   farokra tolja át. Ez pontosan az az állítás, amit a kutatási terv tesz.

## 6. Kép vagy régió: mit címkézünk valójában

*(konzultáció, 5. pont — „ez torzít el minden más mérést, ha rosszul dől el")*

A kiválasztás egy **régióra** mutat. Az annotátor egy **képet** kap. Egy képen négy doboz
is lehet. Három szabály:

- **`box_only`** — csak a kiválasztott doboz kap címkét. A képen minden más **háttérként**
  tanít, beleértve a valódi objektumokat is.
- **`full_image`** — a képen minden annotált objektum megkapja a címkéjét. Nincs
  félcímkézés, viszont képenként több annotációs egységbe kerül.
- **`known_plus_selected`** — a **known** objektumok ingyen vannak (a detektor már tudja
  őket, nem kell hozzá ember), a kiválasztott unknown megkapja a címkét, a többi unknown
  **ignore** lesz, nem háttér.

A `half_labelled_share` oszlop az, amiről ez az egész szól: a háttérként tanított
régiók hány százaléka ül valójában egy valódi annotált objektumon.

In [ ]:
picked = selection.select(pool, selection.ARMS[ARMS[0]], budget=BUDGET_PER_TASK,
                          rounds=ROUNDS_PER_TASK, n_known=len(protocol.TASK1),
                          partition=partition)
rows = []
for policy in labelling.POLICIES:
    annotation = labelling.annotate(pool, picked, policy=policy,
                                    known_classes=protocol.TASK1)
    rows.append(annotation.summary() | {
        "half_labelled_share": labelling.half_labelling_rate(annotation, pool),
        "cost_vs_budget": annotation.oracle_cost / BUDGET_PER_TASK,
        "supervision_per_oracle_unit": annotation.labelled.size / annotation.oracle_cost,
    })
print(table(rows, digits=3))

### A válasz

`known_plus_selected` **ugyanannyiba kerül, mint a `box_only`** — mert a known objektumok
felcímkézéséhez nem kell ember —, **nulla félcímkézéssel**, és **négy-hatszor annyi
felügyeletet** ad a tanításnak ugyanazért az árért.

Ez egybevág azzal, amit a korábbi GPU-futás már megmutatott: amikor a képen amúgy meglévő
task-1 annotációkat visszatettük, a felejtés **27 pontról 2,7-re** esett — replay nélkül.
A memória nem hiányzott; eldobtuk azt, ami ott volt.

> **A GPU-oldali leképezés.** A PROB tanító-loaderében ez a `--supervision-mode`
> kapcsoló: `ft` megtartja az előző taszkok dobozait a kiválasztott képeken,
> `train` eldobja őket. A `box_only` a `train`, a másik kettő az `ft`. Egy valódi
> doboz-szintű szabályhoz szűrt XML-eket kellene írni; ez a következő lépés, nem
> ennek a futásnak a része.

## 7. Replay: mit adunk vissza a régiekből

*(konzultáció, 4. pont — a kutatási terv B kontribúciója)*

$$ m_c \;\propto\; n_c^{\alpha}, \qquad \sum_c m_c = M $$

- $\alpha = 0$: osztályonként egyenlő — a mai standard;
- $\alpha = 1$: mérettel arányos — head-favorizáló;
- $\alpha < 0$: tail-favorizáló.

Három külön kísérleti tengely, és itt három külön paraméter: a memória **mérete**, a
**szabály** ($\alpha$), és hogy taszkonként **újraosztjuk-e** vagy visszük tovább.

Az alábbi tábla a t1 tizenkilenc osztályára mutatja, mit jelent ez konkrétan — a
legritkább osztály 1 294 objektumból tanult, a leggyakoribb 262 465-ből.

In [ ]:
counts = protocol.load_train_counts()
task1_sizes = {name: counts[name] for name in protocol.TASK1}
order = sorted(task1_sizes, key=task1_sizes.get)
rows = []
for name, spec in replay.ARMS.items():
    if spec["total"] == 0:
        continue
    allocation = replay.allocate(task1_sizes, total=spec["total"], alpha=spec["alpha"])
    rows.append({
        "arm": name, "alpha": spec["alpha"], "allocated": sum(allocation.values()),
        "rarest (motorbike-scale)": allocation.get(order[0], 0),
        "median class": allocation.get(order[len(order) // 2], 0),
        "commonest (person)": allocation.get(order[-1], 0),
    })
print(table(rows))
print("\nAt alpha = 1 the rarest class gets a single exemplar and the commonest gets\n"
      "hundreds. That is the failure the research plan predicts for head-favouring\n"
      "allocation, and `minimum=1` is what stops it from reaching zero.")

## 8. A szimulált lánc — mit vesz meg a keret taszkonként

Ez még mindig a fagyasztott készleten fut, tehát **detekciós metrikát nem ad**. Azt
mutatja meg, hogyan alakul a kiválasztás összetétele, ahogy a címkézett halmaz nő és a
$D$ tag egyre nehezebben talál újat.

In [ ]:
# `simulate` walks the chain on the frozen pool and never rehearses: it reports
# what a *score* selects, not what training does with it. So the replay arm here
# only records which configuration this table belongs to, and it is named
# explicitly rather than reaching for a singular REPLAY_ARM that the
# experiment-level parameters no longer define.
diagnostic_replay_arm = REPLAY_ARMS[0]

config = runner.CycleConfig(
    n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK, rounds_per_task=ROUNDS_PER_TASK,
    arm=ARMS[0], labelling_policy=LABELLING_POLICY,
    replay_arm=diagnostic_replay_arm,
    replay_reallocate=REPLAY_REALLOCATE, n_clusters=N_CLUSTERS, seed=SEED,
)
results = runner.simulate(pool, config, chain=chain, partition=partition)
keep = ["task", "new_class", "asked", "objects", "head", "medium", "tail",
        "classes_seen", "label_labelled", "label_oracle_cost", "label_half_labelled"]
print(table([{k: r.flat()[k] for k in keep} for r in results], digits=3))

## 9. A valódi lánc GPU-n

Innentől a PROB súlyai ténylegesen frissülnek, és a PROB saját kiértékelője ad számot.
`RUN_GPU = False` mellett ez a rész kihagyódik.

**Amire szükség van a Drive-on** (`DRIVE_ROOT`):

```
OWL/
  checkpoints/SOWODB/t1.pth              a PROB publikált t1 checkpointja (478 MB)
  data/owdetr_pool_annotations.tar.gz    a jelöltképek VOC-XML annotációi
  data/owdetr_test_annotations.tar.gz    a teszt-annotációk (a repóban is benne van)
```

A képek nem kellenek a Drive-ra: a notebook a COCO-ról tölti le őket igény szerint,
csak azokat, amiket a kiválasztás megnyit.

**Költség.** A kiértékelés a drága rész, nem a tanítás — a teljes 4 952 képes teszt
checkpointonként ~32 perc. Ezért a lánc egy **közös, csökkentett** teszthalmazon mér,
ami minden deklarált osztályból megtartja a képeket (osztályonként legfeljebb
`EVAL_MAX_PER_CLASS`-t), és determinisztikusan mintavételez maradékot. Az így kapott
previous-class mAP **mintabecslés**, publikált teljes-teszt számokkal nem
összehasonlítható — armok között viszont igen, mert mind ugyanazon a halmazon fut.

Minden hívás **újraindítható**: ha a kimenet már megvan, a hívás kimarad. Egy megszakadt
Colab-session ott folytatja, ahol abbahagyta.

### Előellenőrzés — mi van meg a Drive-on, mi nincs

**Ezt futtasd le először,** mielőtt bármi hosszút indítanál. Megnézi, hol vannak a
szükséges fájlok — több szokásos helyen keres, a `DAOWOD` mappát is beleértve, ha korábbi
futásokból már ott van valami —, és kiírja, mi hiányzik. Semmit nem tölt le és nem tanít.

Ha egy sor `MISSING`, az alatta lévő magyarázat megmondja, honnan kell odatenni.

In [ ]:
# Every name the later cells read is assigned here on both branches. A cell that
# reads a name the other branch never set fails with NameError instead of the
# message it was supposed to print, which is exactly the wrong way round.
PREFLIGHT_OK = False
DRIVE = CHECKPOINT = POOL_ANNOTATIONS = TEST_ANNOTATIONS = None

if not RUN_GPU:
    print("RUN_GPU = False — nothing to check. Set it True and rerun this cell.")
else:
    from google.colab import drive as _drive
    _drive.mount("/content/drive", force_remount=False)

    DRIVE = Path(DRIVE_ROOT)
    MY = Path("/content/drive/MyDrive")
    # look in the new folder first, then in the one earlier runs used
    SEARCH_ROOTS = [Path(DRIVE_ROOT), MY / "OWL", MY / "DAOWOD"]

    def locate(*relative_names):
        """First existing match for any of these relative paths, in any root."""
        for root in SEARCH_ROOTS:
            for name in relative_names:
                candidate = root / name
                if candidate.exists():
                    return candidate
        return None

    # the checkpoint is 478 MB and has to be uploaded once; the annotations ship
    # with the repository, so only fall back to Drive if the clone is incomplete
    CHECKPOINT = locate("checkpoints/SOWODB/t1.pth", "checkpoints/t1.pth", "t1.pth")

    STAGING = ROOT / "data" / "staging"
    POOL_ANNOTATIONS = (
        STAGING / "owdetr_pool_annotations.tar.gz"
        if (STAGING / "owdetr_pool_annotations.tar.gz").exists()
        else locate("data/owdetr_pool_annotations.tar.gz", "data/daowod_pool.tar.gz")
    )
    TEST_ANNOTATIONS = STAGING / "owdetr_test_annotations.tar.gz"

    checks = [
        ("PROB t1 checkpoint (478 MB)", CHECKPOINT,
         "Copy exps/SOWODB/PROB/t1.pth from the AI_SSD to "
         f"{DRIVE_ROOT}/checkpoints/SOWODB/t1.pth — this is the only upload needed."),
        ("candidate annotations", POOL_ANNOTATIONS,
         "Ships with the repository. If it is missing, rebuild it on the laptop with "
         "tools/build_pool_annotations.py and upload it to " f"{DRIVE_ROOT}/data/"),
        ("test annotations", TEST_ANNOTATIONS,
         "Ships with the repository — if this is missing the clone is broken."),
    ]
    print(f"{'what':34s} {'status':8s} where")
    print("-" * 100)
    missing = []
    for label, path, remedy in checks:
        if path is None or not Path(path).exists():
            print(f"{label:34s} {'MISSING':8s} → {remedy}")
            missing.append(label)
        else:
            size = path.stat().st_size / 1e6
            print(f"{label:34s} {'ok':8s} {path}  ({size:.0f} MB)")

    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True,
                         check=False)
    print("-" * 100)
    print("GPU:", gpu.stdout.strip() or "NONE — pick a GPU runtime under Runtime > Change runtime type")

    # A bare `raise SystemExit` only stops *this* cell in Colab; the next one runs
    # anyway. So the verdict is a variable the later cells check, not an exception.
    PREFLIGHT_OK = not missing and bool(gpu.stdout.strip())
    if missing:
        print(f"\nCANNOT START: {missing}")
        print("Fix the lines marked MISSING above, then rerun this cell.")
    elif not gpu.stdout.strip():
        print("\nCANNOT START: no GPU. Runtime > Change runtime type > T4 GPU, "
              "then rerun this cell.")
    else:
        print("\nEverything is in place. Run the next cells.")

In [ ]:
if not RUN_GPU:
    print("RUN_GPU = False — the GPU chain is skipped.\n"
          "Everything above ran on the committed PROB pass and is complete on its own.")
    gpu_results = []
else:
    assert PREFLIGHT_OK, (
        "The preflight cell above did not pass. Read its output: it names what is "
        "missing and where to put it. Nothing below can work until it says "
        "'Everything is in place.'"
    )

    DATA = Path("/content/data/OWOD")
    WORK = DRIVE / "work"
    WORK.mkdir(parents=True, exist_ok=True)
    (DATA / "ImageSets" / "OWDETR").mkdir(parents=True, exist_ok=True)

    PROB = bridge.ensure_checkout(Path("/content/PROB"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROB / "requirements.txt")], check=False)

    # Deformable-DETR's multi-scale attention is a compiled CUDA extension. PROB
    # falls back to a pure-PyTorch implementation when it is absent, which is
    # correct but several times slower — at chain length nine that is the
    # difference between one evening and three. Build it once per runtime; the
    # build takes a few minutes and is cached under /content.
    # Whether the extension is importable has to be asked in a *fresh*
    # interpreter. Asking in this one is unreliable: a wheel installed a moment
    # ago is not visible until the import caches are invalidated, so the obvious
    # check reports a working build as failed and you budget three times the
    # wall clock for nothing.
    def msda_available() -> bool:
        probe = subprocess.run(
            [sys.executable, "-c", "import MultiScaleDeformableAttention"],
            capture_output=True, text=True, check=False,
        )
        return probe.returncode == 0

    MSDA_BUILT = msda_available()
    if MSDA_BUILT:
        print("MultiScaleDeformableAttention: already built")
    else:
        # ninja is what the extension's setup.py wants for a parallel build, and
        # its absence is the most common reason this fails on a fresh runtime.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"],
                       check=False)
        nvcc = subprocess.run(["which", "nvcc"], capture_output=True, text=True,
                              check=False).stdout.strip()
        print(f"building MultiScaleDeformableAttention (a few minutes) — nvcc: "
              f"{nvcc or 'NOT FOUND, which will fail the build'}")
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-build-isolation", "."],
            cwd=PROB / "models" / "ops", capture_output=True, text=True, check=False,
        )
        import importlib
        importlib.invalidate_caches()
        MSDA_BUILT = msda_available()
        if MSDA_BUILT:
            print("MultiScaleDeformableAttention: built")
        else:
            log = (build.stdout + "\n" + build.stderr).strip().splitlines()
            errors = [line for line in log
                      if any(token in line.lower()
                             for token in ("error", "fatal", "not found", "no such",
                                           "failed", "undefined"))]
            print("\n*** MultiScaleDeformableAttention: BUILD FAILED ***")
            print("PROB falls back to pure PyTorch. Correct, but the detector pass and\n"
                  "the training both run several times slower — a 2000-image predict\n"
                  "measured 23 min on the fallback against roughly 8 min built. Over\n"
                  "three arms that is the difference between three sessions and six.\n")
            print("What the build actually said:")
            for line in (errors or log)[-25:]:
                print("   ", line[:160])
            print("\nPaste those lines if you want them diagnosed. The chain will run "
                  "either way.")

    # the preflight cell already found these; extract both into one Annotations/ tree
    for source in (POOL_ANNOTATIONS, TEST_ANNOTATIONS):
        with tarfile.open(source) as handle:
            handle.extractall(DATA)
    print("annotations extracted:", len(list((DATA / "Annotations").glob("*.xml"))))

    declared = [t.new_class for t in chain[1:]]
    subset = evaluation_subset.from_archive(
        ROOT / "data" / "staging" / "owdetr_test_annotations.tar.gz",
        declared, seed=SEED, remainder_multiplier=EVAL_REMAINDER_RATIO,
        max_per_class=EVAL_MAX_PER_CLASS,
    )
    # The split name decides which annotation filter PROB applies, so it
    # lives in owl rather than here: a notebook cell saved before a rename
    # would carry the old value, and owl is re-cloned every run.
    TEST_SET = evaluation_subset.SHARED_TEST_SET
    evaluation_subset.write_image_set(
        DATA / "ImageSets" / "OWDETR" / f"{TEST_SET}.txt", subset)
    print(f"shared evaluation split: {len(subset.image_ids)} images, "
          f"~{len(subset.image_ids) * 32 / 4952:.0f} min per checkpoint")
    print("declared-class objects in play:", dict(subset.object_counts))

    candidate_index = json.loads(
        (ROOT / "data" / "reference" / "per_image_class_counts.json").read_text())
    print(f"candidate pool: {len(candidate_index)} unlabelled images")

    # The exemplar memory rehearses on knowledge the model already had, which
    # here is the split t1.pth was trained on. That pool is a separate artefact
    # from the candidate pool on purpose: rehearsing on what the selector just
    # bought would measure the arm's own acquisitions, not retention. If it is
    # not in the repository the run continues WITHOUT rehearsal rather than
    # quietly substituting the candidate pool — see tools/build_replay_index.py.
    replay_index_path = ROOT / "data" / "reference" / "t1_replay_class_counts.json"
    replay_archive = ROOT / "data" / "staging" / "owdetr_replay_annotations.tar.gz"
    if replay_index_path.exists():
        replay_index = json.loads(replay_index_path.read_text())
        if replay_archive.exists():
            with tarfile.open(replay_archive) as handle:
                handle.extractall(DATA)
        print(f"replay pool: {len(replay_index)} task-1 images")
    else:
        replay_index = None
        REPLAY_ARMS = ("none",)
        print("\n*** NO REPLAY POOL — running the no-rehearsal baseline ***")
        print(f"{replay_index_path.name} is not in the repository, so there is no\n"
              "legitimate old-data pool to rehearse on. The candidate pool is not a\n"
              "substitute. REPLAY_ARMS is forced to ('none',) for this run — a real\n"
              "arm, recorded in each workspace's fingerprint, so nothing can later\n"
              "be mixed with a run that did rehearse.\n"
              "Build the pool once with tools/build_replay_index.py --help\n")

    prob_bridge = bridge.Bridge(prob_root=PROB, data_root=DATA,
                                log_dir=WORK / "logs", num_workers=2, seed=SEED)
    print(prob_bridge.check())

    # Price the session with what is actually installed, not with what was hoped.
    slowdown = 1.0 if MSDA_BUILT else 3.0
    per_task = (23.1 / 3.0 * slowdown                      # predict, measured
                + 32.0 * slowdown                          # train, measured shape
                + 2 * len(subset.image_ids) / 1000 * 6.5)  # evaluate, two passes
    total = per_task * (len(chain) - 1) * len(ARMS) / 60
    print(f"\nEXPECTED COST with the {'compiled kernel' if MSDA_BUILT else 'PYTORCH FALLBACK'}: "
          f"~{per_task:.0f} min per task, ~{total:.1f} h for {len(ARMS)} arms "
          f"x {len(chain) - 1} tasks.")
    print(f"TIME_BUDGET_MINUTES is {TIME_BUDGET_MINUTES}, so expect about "
          f"{max(1, round(total * 60 / TIME_BUDGET_MINUTES))} Run all(s). "
          "Each one resumes where the last stopped.")
    if not MSDA_BUILT and total > 20:
        print("\nThat is a lot of sessions. MINIMAL_CHAIN = True in the parameters "
              "cell\ncuts it to three incremental tasks — head plus two tail classes — "
              "which\nstill answers the plan's question, just over a shorter chain.")

### Képek behúzása

Csak azok a képek kellenek, amiket a lánc tényleg megnyit: a közös teszthalmaz itt, a
jelöltképek pedig taszkonként, közvetlenül a detektor-átfutás előtt. A jelöltek a COCO
`train2017`-jében vannak, a teszt-képek a `val2017`-ben, és az azonosítóból nem derül ki,
melyik — ezért mind a kettőt megpróbálja.

**Költség.** Taszkonként `CANDIDATE_IMAGES` kép, egyenként ~160 KB, 32 szálon: 4 000 képnél
néhány perc és ~600 MB. A már letöltötteket kihagyja, tehát egy újraindított futás nem
tölti le újra őket.

Ha egy kép nem elérhető, **kiesik, és nem viszi magával a futást** — a detektor a lemezről
olvas, és az első hiányzó fájlon elhasal, szóval a listát a valóban meglévő képekre kell
szűkíteni, nem feltételezni.

In [ ]:
if RUN_GPU:
    from concurrent.futures import ThreadPoolExecutor

    JPEG = DATA / "JPEGImages"
    JPEG.mkdir(parents=True, exist_ok=True)

    # The candidate images live in COCO's train2017 and the test images in
    # val2017, and nothing tells you which from the id alone — so try both. The
    # detector reads JPEGs off disk and dies on the first missing one, which is
    # why this returns the ids that actually arrived rather than assuming.
    COCO_SPLITS = ("train2017", "val2017")

    def fetch_images(image_ids, splits=COCO_SPLITS, workers=32):
        """Download whatever is not on disk yet. Returns the usable ids."""

        image_ids = [str(value) for value in image_ids]
        wanted = [i for i in image_ids if not (JPEG / f"{i}.jpg").exists()]

        def fetch(image_id):
            target = JPEG / f"{image_id}.jpg"
            for split in splits:
                subprocess.run(
                    ["curl", "-sfL", "--retry", "2", "-o", str(target),
                     f"http://images.cocodataset.org/{split}/{image_id}.jpg"],
                    check=False,
                )
                if target.exists() and target.stat().st_size > 0:
                    return
            target.unlink(missing_ok=True)

        if wanted:
            with ThreadPoolExecutor(max_workers=workers) as pool_exec:
                list(pool_exec.map(fetch, wanted))

        available = [i for i in image_ids if (JPEG / f"{i}.jpg").exists()]
        if len(available) < len(image_ids):
            print(f"    {len(image_ids) - len(available)} of {len(image_ids)} images "
                  "unavailable from COCO; dropped")
        return available

    # the shared evaluation split, once — the chain fetches its own candidates
    ready = fetch_images(subset.image_ids)
    print(f"test images ready: {len(ready)} of {len(subset.image_ids)}")
    assert len(ready) > 0.95 * len(subset.image_ids), (
        "Too many test images failed to download. Check the runtime's network "
        "before spending GPU time on a chain that cannot be evaluated."
    )

In [ ]:
if RUN_GPU:
    from dataclasses import replace

    # `arm` and `replay_arm` are the two things that vary, so neither is set
    # here: the loop below supplies both, and there is exactly one place each
    # comes from.
    base_config = runner.CycleConfig(
        n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK, rounds_per_task=ROUNDS_PER_TASK,
        candidate_images_per_task=CANDIDATE_IMAGES,
        proposals_per_image=PROPOSALS_PER_IMAGE,
        labelling_policy=LABELLING_POLICY,
        replay_reallocate=REPLAY_REALLOCATE, epochs=EPOCHS,
        learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
        n_clusters=N_CLUSTERS, seed=SEED,
    )

    # Every (selection, replay) pair is its own run and its own workspace. The
    # replay arm has to be in the path: this study runs the same selection arm
    # several times, and one workspace per selection arm would put two
    # experiments in one directory — which the fingerprint guard then correctly
    # refuses, after the download.
    scheduled = [(arm, replay_arm) for arm in ARMS for replay_arm in REPLAY_ARMS]
    planned = [f"{arm}__{replay_arm}" for arm, replay_arm in scheduled]

    # TIME_BUDGET_MINUTES is per run, not shared. Sharing it truncates whichever
    # run goes second: at the measured V3 cost a five-task chain is ~263 min, so
    # one 420-minute pot leaves the second run 157 min and it stops after three
    # tasks — two arms that cannot be compared. Each run therefore gets its own
    # cap, and the session ceiling below is what stops scheduling.
    session_ceiling = TIME_BUDGET_MINUTES * len(planned)
    by_arm, spent = {}, 0.0
    for arm in ARMS:
        for replay_arm in REPLAY_ARMS:
            run = f"{arm}__{replay_arm}"
            if spent >= session_ceiling:
                print(f"\n[{run}] not started: the session has used {spent:.0f} of its "
                      f"{session_ceiling:.0f} min ceiling. Run all again to continue.")
                continue
            remaining = TIME_BUDGET_MINUTES
            print(f"\n{'=' * 70}\n{run} — {remaining:.0f} min for this run "
                  f"({spent:.0f}/{session_ceiling:.0f} min used this session)"
                  f"\n{'=' * 70}")
            by_arm[run] = runner.run_chain(
                prob_bridge, replace(base_config, arm=arm, replay_arm=replay_arm),
                workspace=WORK / run,
                candidate_index=candidate_index,
                replay_index=replay_index,
                # where the filtered replay annotations and their image links go.
                # It is PROB's own --data-root, because that is the one Annotations/
                # directory its loader resolves image ids against.
                replay_root=DATA,
                start_checkpoint=CHECKPOINT,
                test_set=TEST_SET, chain=chain,
                time_budget_minutes=remaining,
                prepare_images=fetch_images,
            )
            spent = prob_bridge.cost_report()["total"]
            done = len(by_arm[run])
            print(f"[{run}] {done} of {len(chain) - 1} tasks; {spent:.0f} min used "
                  f"this session of {session_ceiling:.0f}")

    gpu_results = by_arm.get(planned[0], []) if planned else []
    print("\ncost:", prob_bridge.cost_report())
    finished = [a for a, r in by_arm.items() if len(r) == len(chain) - 1]
    partial = [a for a, r in by_arm.items() if 0 < len(r) < len(chain) - 1]
    print("complete runs:", finished or "none")
    if partial:
        print("partial runs:", partial, "— Run all again to finish them")
    if len(finished) < len(planned):
        print(f"not yet complete: {[r for r in planned if r not in finished]}")

## 10. Az eredménytáblázat

Taszkonként hat szám. Az utolsó a legfontosabb: a **csereárfolyam** — hány régi mAP-pontot
fizetünk egy új mAP-pontért. A teljes t2-felügyelet **0,20**-at fizet. Ha egy futás 70-et
fizet, akkor az nem csere, hanem veszteség.

In [ ]:
if RUN_GPU and by_arm:
    for arm, rows in by_arm.items():
        if not rows:
            continue
        print(f"\n{'=' * 78}\n{arm}\n{'=' * 78}")

        headline = ["task", "oracle_cost_so_far", "U_Recall_tail", "U_Recall_medium",
                    "U_Recall_head", "U_Recall_all", "unknown_objects_tail"]
        print("annotation efficiency: unknown recall by frequency group")
        print(table([{k: r.flat().get(k) for k in headline} for r in rows]))

        learned = ["task", "new_class", "known_mAP50", "prev_mAP50", "new_mAP50",
                   "forgetting", "exchange_rate"]
        print("\nwhat was learned and what was lost")
        print(table([{k: r.flat().get(k) for k in learned} for r in rows]))

        grouped = ["task", "mAP50_head", "mAP50_medium", "mAP50_tail"]
        print("\nretention by frequency group")
        print(table([{k: r.flat().get(k) for k in grouped} for r in rows]))

        spend = ["task", "asked", "images_opened", "images_trainable",
                 "images_no_supervision", "images_from_earlier_tasks",
                 "target_objects_in_images", "replay_images"]
        print("\nwhere the annotation budget went")
        print(table([{k: r.flat().get(k) for k in spend} for r in rows]))

    # ---- the comparison the thesis is about -------------------------------
    #
    # The plan's prediction, stated as a table: at the same oracle cost, does
    # distribution-aware selection find more of the tail? Arms are only
    # comparable where all of them reached — a longer chain against a shorter one
    # is not a result.
    depth = min(len(rows) for rows in by_arm.values() if rows)
    if len(by_arm) > 1 and depth:
        print(f"\n{'=' * 78}\ntail U-Recall at equal oracle cost — the plan's prediction"
              f"\n{'=' * 78}")
        comparison = []
        for index in range(depth):
            row = {"task": next(iter(by_arm.values()))[index].task,
                   "oracle_cost": next(iter(by_arm.values()))[index]
                   .flat().get("oracle_cost_so_far")}
            for arm, rows in by_arm.items():
                row[arm] = rows[index].flat().get("U_Recall_tail")
            comparison.append(row)
        print(table(comparison))
        print(f"\nCompared over the {depth} task(s) every arm reached. Higher is better; "
              "the\nprediction is that the first column beats the others at the same cost.")
    elif len(by_arm) > 1:
        print("\nNo arm finished a task yet, so there is nothing to compare.")
    else:
        print("\nOnly one arm has run. Run all again to add the baselines — a single "
              "arm's\nnumbers have nothing to be measured against.")

elif not RUN_GPU:
    print("No GPU results in this session. Reference points from earlier measured runs:\n")
    root = ROOT / "data" / "reference" / "measured"
    reference = []
    for label, filename, baseline in (
        ("full t2 supervision (ceiling)", "full_t2_supervision_metrics.json", 73.649),
        ("random, 600 regions",           "random_b600_metrics.json",         73.649),
        ("distribution-aware, 600",       "mult_prior_shrunk_b600_metrics.json", 73.649),
        ("objectness prior, 600",         "objectness_prior_b600_metrics.json",  73.649),
    ):
        path = root / filename
        if not path.exists():
            continue
        evaluation = metrics.from_bridge_metrics(path)
        row = metrics.task_row(evaluation, task="t2", new_class=None,
                               previous_baseline=baseline)
        row = {"run": label} | row
        row["exchange_rate"] = metrics.exchange_rate(row)
        reference.append(row)
    print(table(reference, digits=3))
    print("\nThese are the twenty-classes-at-once runs. The exchange rate is what the\n"
          "one-class-per-task chain exists to fix.")

## Mit lehet és mit nem lehet állítani ebből

**Lehet.**
- Melyik pontszám mennyi valódi ismeretlent és mennyi *tail*-objektumot vesz meg azonos
  orákulum-költségen. Ez a fagyasztott készleten mérhető, minden arm ugyanazon a
  geometrián fut.
- Hogy a körökre bontás csak azoknak az armoknak segít, amelyeknek van mit frissíteniük.
- Hogy a három címkézési szabály mennyibe kerül, és mennyi félcímkézést okoz.
- Hogy a known-szennyezés orákulum nélkül becsülhető, és mennyire pontosan.

**Nem lehet.**
- Hogy egyik arm kevesebbet *felejt*, mint a másik — ez csak a valódi detektoron mérhető,
  és a fagyasztott szimuláció ezt bizonyítottan **fordítva** rangsorolja.
- Publikált PROB-számokkal összevetni a csökkentett teszthalmazon mért mAP-ot.
- Szignifikanciát három magból. Három mag előjelpróbát bír el, p-értéket nem.

**A hatókör.** Egy benchmark (S-OWODB), egy alapmodell (PROB), és a fagyasztott gerinc
miatt a dobozok soha nem javulnak: az unknown-lefedettség felülről korlátos azzal, amit
`t1.pth` egyáltalán javasol (a jelölthalmaz ismeretlen objektumainak 13,4%-a).